# 5. Extended scenes

A **scene** gathers what an instrument observes: point sources (stars, planets) and diffuse objects
(disks, nebulae, backgrounds, maps). Extended scenes are incoherent: their image is the sum of the images of
their points. LUMOS images them with one of three strategies, chosen automatically and logged:

- **isoplanatic**: the scene convolved by the PSF, for shift-invariant systems;
- **anisoplanatic**: PSFs on a grid of field points, interpolated;
- **decomposition**: every pixel becomes a point source, compulsory for coronagraphs and interferometers.

In [ ]:
import logging

from astropy import units as u

import lumos
from lumos import Context, plotting
from lumos.elements.coronagraphs import VortexCoronagraph
from lumos.elements.optics import FreeSpace, ThinLens
from lumos.elements.telescopes import Telescope
from lumos.fields import Grid
from lumos.plotting import style
from lumos.scenes import ImagingSystem, Scene, image_scene
from lumos.scenes.procedural import GaussianRandomField

style.apply()
logging.basicConfig(level=logging.INFO, format="%(name)s: %(message)s")

## A debris disk behind a coronagraph

The preset `debris_disk.lumos` holds a star and an inclined ring 1000 times fainter, decomposed into point
sources by the `SceneToSources` element and imaged through an 8 m telescope and a vortex coronagraph.

In [ ]:
ctx = lumos.io.load("presets/debris_disk.lumos")
result = ctx.run(ctx.meta["outputs"], archive="debris-disk")
fig, axes = style.figure("double", ncols=2, aspect=0.45)
plotting.plot_radiance(result["sky.radiance"], ax=axes[0])
plotting.plot_psf(
    result["image.out"], ax=axes[1], diameter=8 * u.m, focal_length=100 * u.m, dynamic_range=1e4
)
axes[0].set_title("scene")
axes[1].set_title("coronagraphic image")
result.archive.save_figure(fig, "debris_disk");

## The same scene with `image_scene`

`image_scene` probes the system: an off-axis PSF is compared with the on-axis one. The coronagraph removes
the on-axis star but not the off-axis light: it is not shift-invariant, so the scene is decomposed.

In [ ]:
lam, d, f = 1.6e-6, 8.0, 100.0
resolution = lam / d


def instrument(coronagraph):
    ctx = Context(seed=1)
    ctx.observation.band = (1.59 * u.um, 1.61 * u.um)
    ctx.observation.n_wavelengths = 1
    chain = ctx.add(Telescope(name="telescope", diameter=d * u.m, n_pixels=128, oversize=1.1))
    if coronagraph:
        chain = chain >> VortexCoronagraph(name="vortex", charge=2, pupil_diameter=d * u.m)
    chain = chain >> ThinLens(name="lens", focal_length=f * u.m)
    grid = Grid(64, dx=lam * f / d / 2.5 * u.m)
    chain >> FreeSpace(name="image", distance=f * u.m, method="fresnel_mft", output_grid=grid)
    return ctx


disk = ctx.run(["sky.radiance"])["sky.radiance"]
images = {}
for coronagraph in (False, True):
    system = ImagingSystem(instrument(coronagraph), entrance="telescope.in", exit="image.out")
    images[coronagraph] = image_scene(disk, system)
    print(coronagraph, images[coronagraph].strategy, "-", images[coronagraph].reason)

In [ ]:
fig, axes = style.figure("double", ncols=2, aspect=0.45)
for ax, coronagraph in zip(axes, (False, True), strict=True):
    plotting.plot_psf(
        images[coronagraph].to_wavecube(), ax=ax, diameter=d * u.m, focal_length=f * u.m, log=False
    )
    ax.set_title(f"{'with' if coronagraph else 'without'} coronagraph ({images[coronagraph].strategy})")
result.archive.save_figure(
    fig, "strategies", data={"without": images[False].power, "with": images[True].power}
);

## A nebula

A random texture with a power-law spectrum stands for a nebula; imaged by a plain telescope, the system is
shift-invariant and the scene is convolved by the PSF.

In [ ]:
sky = Context(seed=7)
sky.observation.band = (1.59 * u.um, 1.61 * u.um)
scene = sky.add(Scene(name="nebula", n_pixels=64, pixel_scale=0.5 * resolution * u.rad))
scene.add(
    GaussianRandomField(
        name="texture",
        surface_brightness=1e-3 * u.Jy / u.arcsec**2,
        contrast=0.5,
        spectral_index=3.0,
        n_pixels=64,
        pixel_scale=0.5 * resolution * u.rad,
    )
)
nebula = sky.run(["nebula.radiance"], archive="nebula")
image = image_scene(
    nebula["nebula.radiance"], ImagingSystem(instrument(False), entrance="telescope.in", exit="image.out")
)
fig, axes = style.figure("double", ncols=2, aspect=0.45)
plotting.plot_radiance(nebula["nebula.radiance"], ax=axes[0])
plotting.plot_psf(image.to_wavecube(), ax=axes[1], log=False)
axes[1].set_title(image.strategy)
nebula.archive.save_figure(fig, "nebula");